In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Load the finalized splits from Day 3
train_df = pd.read_csv('../data/processed/train_split.csv')
val_df = pd.read_csv('../data/processed/val_split.csv')
test_df = pd.read_csv('../data/processed/test_split.csv')

# 2. Define features (X) and target (y)
# We drop 'Date' and 'Units Sold' from features, as 'Units Sold' is what we want to predict
drop_cols = ['Date', 'Units Sold']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['Units Sold']

X_val = val_df.drop(columns=drop_cols)
y_val = val_df['Units Sold']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['Units Sold']

print("Data loaded successfully!")
print("Training Features Shape:", X_train.shape)
print("Validation Features Shape:", X_val.shape)

Data loaded successfully!
Training Features Shape: (47600, 26)
Validation Features Shape: (7500, 26)


In [2]:
# ==========================================
# BASELINE MODEL & EVALUATION FUNCTION
# ==========================================

# 1. Simple Baseline: Predict yesterday's sales (Units_Sold_Lag_1) as today's prediction
baseline_preds = X_val['Units_Sold_Lag_1']

# 2. Define a helper function to evaluate models
def evaluate_model(y_true, y_pred, model_name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- {model_name} Performance ---")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}\n")
    return {"Model": model_name, "MAE": mae, "RMSE": rmse, "R2": r2}

# Evaluate the baseline
baseline_results = evaluate_model(y_val, baseline_preds, model_name="Naive Baseline (Lag 1)")

--- Naive Baseline (Lag 1) Performance ---
MAE:  122.3372
RMSE: 156.3671
R²:   -1.0465



In [4]:
# Check which columns in X_train are non-numeric (object/text type)
non_numeric_cols = X_train.select_dtypes(include=['object']).columns
print("Non-numeric columns found that need to be dropped:", list(non_numeric_cols))

# Automatically drop any text/object columns from X_train, X_val, and X_test
X_train = X_train.select_dtypes(exclude=['object'])
X_val = X_val.select_dtypes(exclude=['object'])
X_test = X_test.select_dtypes(exclude=['object'])

print("Updated Training Features Shape:", X_train.shape)

Non-numeric columns found that need to be dropped: ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']
Updated Training Features Shape: (47600, 20)


C:\Users\mayank chhipa\AppData\Local\Temp\ipykernel_26168\3192229126.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  non_numeric_cols = X_train.select_dtypes(include=['object']).columns


In [ ]:
from sklearn.linear_model import LinearRegression

# 1. Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_val)
lr_results = evaluate_model(y_val, lr_preds, model_name="Linear Regression")

--- Linear Regression Performance ---
MAE:  8.5039
RMSE: 13.5703
R²:   0.9846



In [6]:
from sklearn.ensemble import RandomForestRegressor

# 2. Train Random Forest Regressor (n_estimators=100 for a solid balance of speed and performance)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)
rf_results = evaluate_model(y_val, rf_preds, model_name="Random Forest Regressor")

--- Random Forest Regressor Performance ---
MAE:  7.3641
RMSE: 8.6434
R²:   0.9937



In [7]:
# Combine all results into a summary comparison table
results_df = pd.DataFrame([baseline_results, lr_results, rf_results])
display(results_df)

,Model,MAE,RMSE,R2
0,Naive Baseline (Lag 1),122.337200,156.367123,-1.046468
1,Linear Regression,8.503902,13.570321,0.984587
2,Random Forest Regressor,7.364111,8.643418,0.993747
